# Create sample data and populate Feature Store

Run inside the application checkout. Fill in the bucket and exact IAM role ARN. Leave the metadata path empty for the included demo schema, or supply your own.
No container build or CDK deployment is required.

The sample repeats the five supplied reference rows. It tests field names, types and data movement;
it does not reproduce realistic relationships or measure model quality. Shared keys are synthetic.
This notebook creates **offline-only** groups, ingests one delivery, and reads it back with Athena.
GetData and batch Preprocess query this store during each pipeline execution and retain their own S3 snapshots.

## 1. Inputs

- Supply the schema file, existing bucket and the role Feature Store can assume to write offline data.
- Set record_id and event_time to the shared key names your model code expects.
- Repeated metadata rows for those keys become one CSV column each; every group receives the same generated keys.
- This notebook only sets up and populates the store; model selection and population/target locations belong to pipeline execution.

In [ ]:
metadata_csv = ""                   # Empty uses the included demo schema; otherwise supply your CSV path.
bucket = ""                        # Existing bucket name, with or without s3://.
role_arn = ""                      # Exact IAM role ARN, including its path.

namespace = "source-dev"
sample_rows = 100
record_id = "customer_id"
event_time = "feature_asof_dt"
sample_asof = "2026-01-01T00:00:00Z" # Fixed synthetic date, not the current date.
type_overrides = {}                # column_name: binary / integer / continuous / string.
athena_workgroup = "primary"       # Existing workgroup; change if your platform supplies one.
kms_key = ""                       # Optional platform-provided KMS key ARN.



## 2. Generate the local sample and inspect its mapping

- Use `column_name` as the CSV header and `table_nm` as the source grouping; retain `column_map`, `type` and `length` in the metadata.
- Row 1 uses `ref_val1`, row 2 uses `ref_val2`, through row 5, then repeats; text, empty fields and category spellings are preserved.
- Review the suggested storage types and override ambiguous columns in cell 1; the numeric metadata type codes are not assumed to define their meaning.
- Save the sample and mapping locally; this cell makes no AWS calls.

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib
import importlib
import json
import sys
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-features/data.py").is_file())
sys.path[:0] = [str(ROOT / "mlops-features"), str(ROOT / "mlops-models/src")]
import mlops_features.metadata as metadata_module
generate_sample = importlib.reload(metadata_module).generate_sample
from mlops_features.feature_mapping import read_mapping, group_columns, group_name
from mlops_features.data import validate_delivery, ingest_csv, read_dataset

metadata_csv = metadata_csv or str(ROOT / "examples/data/feature-metadata.csv")
RUN = ROOT / ".runs/feature-store" / namespace / uuid4().hex
sample = generate_sample(
    metadata_csv, RUN, rows=sample_rows, record_id=record_id, event_time=event_time,
    asof=sample_asof, type_overrides=type_overrides, deduplicate=True,
)
mapping = read_mapping(sample["mapping"], record_id, event_time)
groups = {
    key: {"name": group_name(namespace, key), "columns": columns}
    for key, columns in group_columns(mapping, record_id, event_time).items()
}
validate_delivery(sample["sample"], {"record_id": record_id, "event_time": event_time, "mapping": mapping})
source_frame = pd.read_csv(sample["sample"], dtype=str, keep_default_na=False)
print("Metadata module:", metadata_module.__file__)
print("Local sample:", sample["sample"].resolve())
print("Mapping:", sample["mapping"].resolve())
print("Source column mapping:", sample["metadata"].resolve())
print("Rows:", sample["rows"], "| Features:", sample["features"], "| Source tables:", sample["tables"])
print("Feature groups:", {group["name"]: len(group["columns"]) + 1 for group in groups.values()})
print("Storage types are suggestions from the five reference values. Review before creating groups.")
display(pd.DataFrame(mapping).head(20))
display(source_frame.iloc[:5, :10])


## 3. Connect to AWS and plan the storage paths

- Read the current AWS identity; keep it distinct from the offline writer role supplied above.
- Use one derived prefix in the supplied bucket for offline storage, receipts, configuration and query results.
- Record settings locally; this cell creates no AWS resources.

In [ ]:
import boto3
from botocore.exceptions import ClientError
import time

bucket = bucket.strip().removeprefix("s3://").rstrip("/")
assert bucket and "/" not in bucket and role_arn.strip(), "Fill in bucket and role_arn."
session = boto3.Session()
sm = session.client("sagemaker")
caller = session.client("sts").get_caller_identity()
settings = {
    "namespace": namespace, "account": caller["Account"], "region": session.region_name,
    "bucket": bucket, "prefix": namespace + "/features/", "record_id": record_id,
    "event_time": event_time, "mapping": mapping,
    "mapping_sha256": hashlib.sha256(sample["mapping"].read_bytes()).hexdigest(),
    "groups": groups, "workgroup": athena_workgroup, "kms_key": kms_key,
    "query_output": f"s3://{bucket}/{namespace}/features/queries/",
    "source_metadata": json.loads(sample["metadata"].read_text()),
}
print("Caller:", caller["Arn"])
print("Offline writer role:", role_arn)
print("Storage:", f"s3://{bucket}/{settings['prefix']}")
print("Athena workgroup:", athena_workgroup)
print("Local evidence:", RUN.resolve())


## 4. Create the offline feature groups

- Create one group per source table, splitting tables above 2,497 mapped features to leave room for ID, event time and delivery ID.
- Let Feature Store create its Glue catalog tables and use the returned database/table names.
- Reuse an existing group only when its definitions, keys, role and storage match; schema changes need a new namespace or a reviewed migration.
- Wait for each group before creating the next; permission errors are shown unchanged.

In [ ]:
offline_uri = f"s3://{bucket}/{settings['prefix']}offline/"
for group in groups.values():
    definitions = [{"FeatureName": name, "FeatureType": kind}
                   for name, kind in {**group["columns"], "delivery_id": "String"}.items()]
    storage = {"S3Uri": offline_uri}
    if kms_key:
        storage["KmsKeyId"] = kms_key
    try:
        description = sm.describe_feature_group(FeatureGroupName=group["name"])
    except ClientError as error:
        if error.response["Error"]["Code"] != "ResourceNotFound":
            raise
        response = sm.create_feature_group(
            FeatureGroupName=group["name"], RecordIdentifierFeatureName=record_id,
            EventTimeFeatureName=event_time, FeatureDefinitions=definitions,
            RoleArn=role_arn.strip(), OnlineStoreConfig={"EnableOnlineStore": False},
            OfflineStoreConfig={"S3StorageConfig": storage, "DisableGlueTableCreation": False},
        )
        print("Created:", response["FeatureGroupArn"])
        description = sm.describe_feature_group(FeatureGroupName=group["name"])
    deadline = time.monotonic() + 900
    while description["FeatureGroupStatus"] == "Creating" and time.monotonic() < deadline:
        print("Waiting for:", group["name"], flush=True)
        time.sleep(15)
        description = sm.describe_feature_group(FeatureGroupName=group["name"])
    assert description["FeatureGroupStatus"] == "Created", description
    actual_definitions = {item["FeatureName"]: item["FeatureType"] for item in description["FeatureDefinitions"]}
    assert actual_definitions == {item["FeatureName"]: item["FeatureType"] for item in definitions}, "Existing schema differs."
    assert description["RecordIdentifierFeatureName"] == record_id and description["EventTimeFeatureName"] == event_time
    assert description["RoleArn"] == role_arn.strip(), "Existing offline writer role differs."
    actual_storage = description["OfflineStoreConfig"]["S3StorageConfig"]
    assert actual_storage["S3Uri"].rstrip("/") == offline_uri.rstrip("/"), "Existing offline location differs."
    if kms_key:
        assert actual_storage.get("KmsKeyId") == kms_key, "Existing encryption key differs."
    catalog = description["OfflineStoreConfig"]["DataCatalogConfig"]
    group.update(database=catalog["Database"], table=catalog["TableName"], catalog=catalog["Catalog"])
    print("Ready:", group["name"], "| Table:", group["database"] + "." + group["table"])
(RUN / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print("Saved settings:", (RUN / "settings.json").resolve())


## 5. Read the generated CSV and ingest its fields

- Validate keys and types, then submit each row to its mapped offline groups.
- Keep the same delivery ID if this cell needs a retry; completion is recorded only after every group succeeds.
- Save the receipt locally; run the final readback cell to confirm offline visibility before starting the pipeline.

In [ ]:
receipt = ingest_csv(
    session, settings, sample["sample"], delivery_id=RUN.name,
    upstream_revision="metadata-sha256:" + settings["source_metadata"]["metadata_sha256"],
)
(RUN / "ingestion.json").write_text(json.dumps(receipt, indent=2) + "\n")
print("Completed delivery:", receipt["delivery_id"])
print("Records per group:", receipt["records"])
print("Receipt:", (RUN / "ingestion.json").resolve())


## 6. Save the pipeline's Feature Store configuration

- Publish group names, field mappings and query settings at a new S3 location.
- Give this URI to 03Fv2; it becomes the pipeline's FeatureStoreConfigUri parameter.
- The pipeline resolves a completed delivery and queries it itself. No extracted training CSV is needed here.


In [ ]:
from mlops_features.data import write_once
config_key = settings["prefix"] + "configurations/" + uuid4().hex + "/feature-store.json"
write_once(session.client("s3"), bucket, config_key, settings, kms_key=kms_key)
feature_store_config_uri = f"s3://{bucket}/{config_key}"
print("Set this in 03Fv2 cell 1:")
print("feature_store_config_uri =", repr(feature_store_config_uri))
print("Completed delivery ID:", receipt["delivery_id"])
print("Leave FeatureStoreDeliveryId blank for the most recently completed delivery, or set this ID to replay it.")
print("The next optional cell checks readback here. It is not required to run a published pipeline.")


## 7. Optional: verify the offline values here

- Query the completed delivery and compare returned values with the generated sample.
- Keep this local check separate from pipeline execution; the pipeline performs its own query and snapshot.
- If offline delivery is still pending, the reader waits; retry this cell with the same receipt after a timeout.


In [ ]:
DATASET = RUN / ("readback-" + uuid4().hex)
dataset = read_dataset(session, settings, receipt, DATASET, publish=False)
types = {"String": str, "Integral": "Int64", "Fractional": float}
csv_types = {name: types[kind] for group in groups.values() for name, kind in group["columns"].items()}
original = pd.read_csv(sample["sample"], dtype=csv_types, keep_default_na=False, na_values=[""])
retrieved = pd.read_csv(DATASET / "data.csv", dtype=csv_types, keep_default_na=False, na_values=[""])
original[event_time] = pd.to_datetime(original[event_time], utc=True).map(lambda value: value.isoformat().replace("+00:00", "Z"))
keys = [record_id, event_time]
pd.testing.assert_frame_equal(
    original.sort_values(keys).reset_index(drop=True),
    retrieved.sort_values(keys).reset_index(drop=True),
    check_exact=False, rtol=1e-12, atol=1e-12,
)
print("Verified offline values:", len(retrieved), "rows")
print("Local diagnostic files:", DATASET.resolve())
display(retrieved.iloc[:5, :10])


After setup, use **03Fv2** to publish training and inference, then start executions in Studio.
The data publisher records completion only after every feature group succeeds. Pipelines select a completed delivery
once and save the selected ID, query details and exact consumed rows under their own execution's `data/raw/` prefix.

The included demo schema supplies synthetic regression/classification targets, so both example models work with empty `UniverseDataRoot`. A supplied schema is used as given: the generator does not add labels to it. Training needs target labels either in the selected
data or in a matching population CSV. Set `UniverseDataRoot` to a prefix containing `<ModelId>/universe.csv`;
training expects ID/date/label, while inference needs only ID/date. Empty `UniverseDataRoot` on inference scores
all rows in the selected delivery. Port the model's configured feature names to match the supplied schema.
